# MolGap Phase 8: resumable 3D SchNet on Colab

This notebook is the current PyG SchNet path: ETKDG + MMFF(200) coordinates, Gasteiger charges, and raw B3LYP eV targets. It mirrors `src/molgap/schnet.py` and the Phase 8 500K training configuration.

It never rebuilds the existing 500K cache. The missing 500K top-up cache is built on the 32-core CPU node, then uploaded to Drive. This notebook directly trains the combined 1M 3D set; every checkpoint and result is written to Drive.

In [ ]:
# 0. Mount Drive first, then install runtime dependencies. Select a GPU runtime before running this cell.
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, torch
assert torch.cuda.is_available(), 'No CUDA GPU. Colab: Runtime > Change runtime type > GPU, then reconnect.'
torch_ver = torch.__version__.split('+')[0]
cuda_ver = torch.version.cuda
assert cuda_ver, f'CUDA build of PyTorch required, got {torch.__version__}'
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver.replace('.', '')}.html"
print('torch:', torch.__version__, '| CUDA:', cuda_ver, '| wheel index:', wheel_url)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'torch-geometric', 'rdkit', 'pandas', 'scikit-learn', 'tqdm', 'psutil'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster', '-f', wheel_url])
print('GPU:', torch.cuda.get_device_name(0), '| PyG dependencies installed')

In [ ]:
# 1. Resolve the two files you uploaded from the mounted Drive.
from pathlib import Path
import json, os, shutil, time, random, hashlib
import numpy as np
import pandas as pd

DRIVE = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE / 'MolGap'
RAW_ROOT = PROJECT_ROOT / 'raw_data'
RESULTS_ROOT = PROJECT_ROOT / 'results'
CHECKPOINTS_ROOT = PROJECT_ROOT / 'checkpoints'
RUN_DIR = RESULTS_ROOT / 'molgap_phase8_colab_3d'
CHECKPOINT_DIR = CHECKPOINTS_ROOT / 'molgap_phase8_colab_3d'
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def find_one(root, exact_name, suffix=None):
    hits = sorted(root.rglob(exact_name))
    if not hits and suffix:
        hits = sorted(root.rglob(f'*{suffix}'))
    assert len(hits) == 1, f'Expected exactly one {exact_name} below {root}; found: {hits}'
    return hits[0]

assert PROJECT_ROOT.exists(), f'Missing {PROJECT_ROOT}'
assert RAW_ROOT.exists(), f'Missing {RAW_ROOT}'
assert RESULTS_ROOT.exists(), f'Missing {RESULTS_ROOT}'
assert CHECKPOINTS_ROOT.exists(), f'Missing {CHECKPOINTS_ROOT}'
RAW_CSV = find_one(RAW_ROOT, 'phase8_expansion_1m.csv', '.csv')
BASE_CACHE = find_one(RESULTS_ROOT, 'pyg_3d_graphs_etkdg_expansion_500k.pt', '.pt')
print('raw CSV :', RAW_CSV, f'({RAW_CSV.stat().st_size / 1e6:.1f} MB)')
print('base 3D :', BASE_CACHE, f'({BASE_CACHE.stat().st_size / 1e9:.2f} GB)')
print('results :', RUN_DIR)
print('checkpoints:', CHECKPOINT_DIR)

# This is deliberately lightweight. Full SHA256 of a 1.33 GB cache is optional.
VERIFY_SHA256 = False
if VERIFY_SHA256:
    def sha256(path):
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()
    print('base SHA256:', sha256(BASE_CACHE))

In [ ]:
# 2. Validate the cache and configure the run. No raw CSV is loaded during 500K training.
from torch_geometric.data import Data

SEED = 42
TARGETS = ['homo', 'lumo', 'gap']
DATASET_MODE = 'extend_1m'    # 'reuse_500k' or 'extend_1m'
BUILD_TOPUP = False           # Top-up is built on the 32-core CPU node and uploaded to Drive.
TOPUP_LIMIT = None            # None builds rows 500000:1000000; use 30000 only for a builder smoke.
TOPUP_TAG = 'full' if TOPUP_LIMIT is None else f'n{TOPUP_LIMIT}'
TOPUP_NAME = f'pyg_3d_graphs_etkdg_expansion_1m_topup_{TOPUP_TAG}.pt'
SCHNET_EMBEDDING_NAME = 'schnet_expansion_1m_embeddings.pt'
# A CPU-built tail may be uploaded directly under results/; find it recursively.
TOPUP_CACHE = RUN_DIR / TOPUP_NAME if BUILD_TOPUP else find_one(RESULTS_ROOT, TOPUP_NAME, '.pt')
TRAIN_LIMIT = None            # Direct full 1M training after both graph caches are present.
MAX_EPOCHS = 12               # Matches the original 500K cosine schedule length.
PATIENCE = 25
BATCH_SIZE = 128              # Drop to 64 only on CUDA OOM.
NUM_WORKERS = 2
BUILD_WORKERS = min(8, max(1, (os.cpu_count() or 2) - 1))
WARM_START = None              # Optional compatible .pt checkpoint placed under Drive/results.
RESUME = True

base_graphs = torch.load(BASE_CACHE, weights_only=False)
assert len(base_graphs) == 497578, f'Unexpected base graph count: {len(base_graphs)}'
assert all(hasattr(g, name) for name in ('z', 'pos', 'charges', 'y', 'source_idx') for g in base_graphs[:3])
base_source = torch.cat([g.source_idx.view(-1).cpu() for g in base_graphs]).numpy()
assert np.all(np.diff(base_source) > 0) and base_source.min() == 0 and base_source.max() < 500000
sample_y = torch.cat([g.y for g in base_graphs[:1024]], dim=0).numpy()
assert np.max(np.abs(sample_y[:, 2] - (sample_y[:, 1] - sample_y[:, 0]))) < 1e-5
print(f'base cache OK: {len(base_graphs):,} ETKDG graphs; source_idx=[{base_source.min()}, {base_source.max()}]')
print('top-up cache:', TOPUP_CACHE.name)
print('sample label algebra Gap = LUMO - HOMO: OK')

In [ ]:
# 3. Optional: build ONLY the missing 1M top-up rows. Shards and progress live on Drive.
# Run this cell only with BUILD_TOPUP=True. It never touches BASE_CACHE.
if BUILD_TOPUP and not TOPUP_CACHE.exists():
    import multiprocessing as mp
    from rdkit import Chem
    from rdkit.Chem import AllChem
    from torch_geometric.data import Data
    from tqdm.auto import tqdm

    df = pd.read_csv(RAW_CSV)
    df = df.dropna(subset=TARGETS + ['smiles']).copy()
    df = df[df['gap'] > 0].reset_index(drop=True)
    assert len(df) == 1000000, f'Expected 1,000,000 valid rows, got {len(df)}'
    smiles_col = 'canonical_smiles' if 'canonical_smiles' in df.columns else 'smiles'
    stop = len(df) if TOPUP_LIMIT is None else min(len(df), 500000 + int(TOPUP_LIMIT))
    work_rows = [(i, df.at[i, smiles_col], df.loc[i, TARGETS].to_numpy(np.float32).tolist()) for i in range(500000, stop)]
    del df

    shard_dir = RUN_DIR / 'topup_shards'
    shard_dir.mkdir(exist_ok=True)
    progress_path = RUN_DIR / 'topup_build.progress'
    shard_size = 5000
    start = int(progress_path.read_text()) if progress_path.exists() else 0
    assert 0 <= start <= len(work_rows), f'Bad progress: {start}'

    def build_one(item):
        source_idx, smiles, target = item
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None: return None
            mol_h = AllChem.AddHs(mol)
            for _ in range(2):
                if AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv3()) == 0: break
            else: return None
            try: AllChem.MMFFOptimizeMolecule(mol_h, maxIters=200)
            except Exception: pass
            if mol_h.GetNumAtoms() == 0: return None
            AllChem.ComputeGasteigerCharges(mol_h)
            charges = []
            for atom in mol_h.GetAtoms():
                c = atom.GetDoubleProp('_GasteigerCharge')
                charges.append(0.0 if (c != c) or abs(c) > 1e6 else c)
            return Data(
                z=torch.tensor([a.GetAtomicNum() for a in mol_h.GetAtoms()], dtype=torch.long),
                pos=torch.tensor(mol_h.GetConformer().GetPositions(), dtype=torch.float32),
                charges=torch.tensor(charges, dtype=torch.float32),
                y=torch.tensor(target, dtype=torch.float32).unsqueeze(0),
                source_idx=torch.tensor([source_idx], dtype=torch.long),
            )
        except Exception:
            return None

    buf, shard_idx, processed = [], start // shard_size, start
    context = mp.get_context('fork')
    with context.Pool(BUILD_WORKERS) as pool:
        for graph in tqdm(pool.imap(build_one, work_rows[start:], chunksize=200), total=len(work_rows)-start, desc=f'ETKDG top-up ({BUILD_WORKERS} workers)'):
            processed += 1
            if graph is not None: buf.append(graph)
            if processed % shard_size == 0:
                torch.save(buf, shard_dir / f'shard_{shard_idx:04d}.pt')
                buf.clear(); shard_idx += 1; progress_path.write_text(str(processed))
    if buf: torch.save(buf, shard_dir / f'shard_{shard_idx:04d}.pt')
    progress_path.write_text(str(processed))
    tail_graphs = []
    for path in sorted(shard_dir.glob('shard_*.pt')):
        tail_graphs.extend(torch.load(path, weights_only=False))
    assert len(tail_graphs) > 0 and all(int(g.source_idx) >= 500000 for g in tail_graphs)
    torch.save(tail_graphs, TOPUP_CACHE)
    json.dump({'raw_csv': str(RAW_CSV), 'processed_topup_rows': processed, 'graphs': len(tail_graphs), 'workers': BUILD_WORKERS}, open(RUN_DIR / 'topup_build_report.json', 'w'), indent=2)
    print(f'Top-up complete: {len(tail_graphs):,}/{len(work_rows):,} graphs -> {TOPUP_CACHE}')
elif BUILD_TOPUP:
    print('Top-up cache already exists:', TOPUP_CACHE)
else:
    print('Skipped. Set BUILD_TOPUP=True only after the GPU gate.')

In [ ]:
# 4. Select cache and define the deployed Phase 8 SchNet wrapper.
assert DATASET_MODE in {'reuse_500k', 'extend_1m'}
if DATASET_MODE == 'reuse_500k':
    graphs = base_graphs
else:
    assert TOPUP_CACHE.exists(), 'Build the top-up cache first.'
    topup_graphs = torch.load(TOPUP_CACHE, weights_only=False)
    graphs = base_graphs + topup_graphs
    assert len(torch.unique(torch.cat([g.source_idx.view(-1) for g in graphs]))) == len(graphs)
print(f'active graphs: {len(graphs):,} | mode: {DATASET_MODE}')

import torch.nn as nn
from torch_geometric.nn import global_mean_pool
from torch_geometric.nn.models import SchNet
from torch_geometric.nn.models.schnet import radius_graph

class SchNetWrapper(nn.Module):
    def __init__(self, hidden_channels=192, num_filters=192, num_interactions=6, num_gaussians=50, cutoff=6.0, dropout=0.0, n_targets=3, use_charges=True):
        super().__init__()
        self.schnet = SchNet(hidden_channels=hidden_channels, num_filters=num_filters, num_interactions=num_interactions, num_gaussians=num_gaussians, cutoff=cutoff)
        self.use_charges = use_charges
        if use_charges: self.charge_proj = nn.Linear(1, hidden_channels)
        self.head = nn.Sequential(nn.Linear(hidden_channels, hidden_channels), nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden_channels, hidden_channels // 2), nn.SiLU(), nn.Linear(hidden_channels // 2, n_targets))
    def encode(self, z, pos, batch, charges=None):
        h = self.schnet.embedding(z)
        if self.use_charges and charges is not None: h = h + self.charge_proj(charges.unsqueeze(-1))
        edge_index = radius_graph(pos, r=self.schnet.cutoff, batch=batch, max_num_neighbors=32)
        row, col = edge_index
        edge_weight = (pos[row] - pos[col]).norm(dim=-1)
        edge_attr = self.schnet.distance_expansion(edge_weight)
        for interaction in self.schnet.interactions: h = h + interaction(h, edge_index, edge_weight, edge_attr)
        return global_mean_pool(h, batch)
    def forward(self, z, pos, batch, charges=None):
        return self.head(self.encode(z, pos, batch, charges))

PARAMS = {'hidden_channels': 192, 'num_filters': 192, 'num_interactions': 6, 'num_gaussians': 50, 'cutoff': 6.0, 'dropout': 0.0}
print('SchNet parameters:', PARAMS)

In [ ]:
# 5. Train raw-eV SchNet with epoch-level Drive checkpoints and exact restart support.
from torch_geometric.loader import DataLoader
from sklearn.metrics import mean_absolute_error, r2_score
from contextlib import nullcontext

def forward_batch(model, batch):
    return model(batch.z, batch.pos, batch.batch, charges=getattr(batch, 'charges', None))

def split_indices(n):
    # Matches scripts/phase8/train_encoder.py, not the old normalized Phase 6 notebook.
    idx = np.random.RandomState(SEED).permutation(n)
    n_train, n_val = int(0.8 * n), int(0.1 * n)
    return idx[:n_train], idx[n_train:n_train+n_val], idx[n_train+n_val:]

active_idx = np.arange(len(graphs))
if TRAIN_LIMIT is not None:
    assert 1000 <= TRAIN_LIMIT <= len(graphs)
    active_idx = np.random.RandomState(SEED).permutation(active_idx)[:TRAIN_LIMIT]
active_idx.sort()
active_graphs = [graphs[i] for i in active_idx]
train_i, val_i, test_i = split_indices(len(active_graphs))
train_set = [active_graphs[i] for i in train_i]; val_set = [active_graphs[i] for i in val_i]; test_set = [active_graphs[i] for i in test_i]
del active_graphs
print(f'split: train={len(train_set):,} val={len(val_set):,} test={len(test_set):,} (raw eV)')

device = torch.device('cuda')
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
model = SchNetWrapper(**PARAMS).to(device)
if WARM_START:
    state = torch.load(WARM_START, map_location='cpu', weights_only=True)
    model.load_state_dict(state)
    print('warm-started:', WARM_START)
optimizer = torch.optim.AdamW(model.parameters(), lr=2.1150972021685588e-4, weight_decay=1.4656553886225336e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')
criterion = nn.L1Loss()
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)

tag = f'{DATASET_MODE}_n{len(train_set)+len(val_set)+len(test_set)}'
last_path, best_path = CHECKPOINT_DIR / f'{tag}_last.pt', CHECKPOINT_DIR / f'{tag}_best.pt'
metrics_path, log_path = RUN_DIR / f'{tag}_metrics.json', RUN_DIR / f'{tag}_train_log.csv'
start_epoch, best_val, best_epoch, wait, log_rows = 0, float('inf'), -1, 0, []
if RESUME and last_path.exists():
    payload = torch.load(last_path, map_location='cpu', weights_only=False)
    assert payload['tag'] == tag and payload['params'] == PARAMS, 'Checkpoint config does not match this run.'
    model.load_state_dict(payload['model']); optimizer.load_state_dict(payload['optimizer']); scheduler.load_state_dict(payload['scheduler']); scaler.load_state_dict(payload['scaler'])
    start_epoch, best_val, best_epoch, wait, log_rows = payload['epoch'] + 1, payload['best_val'], payload['best_epoch'], payload['wait'], payload['log']
    print(f'resuming epoch {start_epoch}; best={best_val:.5f}@{best_epoch}')

for epoch in range(start_epoch, MAX_EPOCHS):
    t0 = time.time(); model.train(); total = count = 0
    for batch in train_loader:
        batch = batch.to(device, non_blocking=True); optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            loss = criterion(forward_batch(model, batch), batch.y)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        scaler.step(optimizer); scaler.update(); total += loss.item() * batch.num_graphs; count += batch.num_graphs
    train_mae = total / max(count, 1)
    model.eval(); total = count = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                loss = criterion(forward_batch(model, batch), batch.y)
            total += loss.item() * batch.num_graphs; count += batch.num_graphs
    val_mae = total / max(count, 1); scheduler.step(); improved = val_mae < best_val
    if improved:
        best_val, best_epoch, wait = val_mae, epoch, 0
        torch.save(model.state_dict(), best_path)
    else: wait += 1
    row = {'epoch': epoch, 'train_mae': train_mae, 'val_mae': val_mae, 'best_val_mae': best_val, 'lr': optimizer.param_groups[0]['lr'], 'seconds': time.time()-t0}
    log_rows.append(row); pd.DataFrame(log_rows).to_csv(log_path, index=False)
    torch.save({'tag': tag, 'epoch': epoch, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'best_val': best_val, 'best_epoch': best_epoch, 'wait': wait, 'log': log_rows, 'params': PARAMS}, last_path)
    print(f"ep{epoch:03d} train={train_mae:.5f} val={val_mae:.5f} best={best_val:.5f}@{best_epoch} lr={optimizer.param_groups[0]['lr']:.2e} {row['seconds']:.0f}s{' *' if improved else ''}")
    if wait >= PATIENCE: print('early stop'); break

assert best_path.exists(), 'No checkpoint was produced.'
model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))

In [ ]:
# 6. Evaluate the best checkpoint in eV and save a compact reproducibility record.
model.eval(); pred, true = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            out = forward_batch(model, batch)
        pred.append(out.float().cpu().numpy()); true.append(batch.y.float().cpu().numpy())
pred, true = np.concatenate(pred), np.concatenate(true)
metrics = {}
for i, name in enumerate(['HOMO', 'LUMO', 'Gap']):
    metrics[name] = {'mae_eV': float(mean_absolute_error(true[:, i], pred[:, i])), 'r2': float(r2_score(true[:, i], pred[:, i]))}
metrics['average'] = {'mae_eV': float(np.mean([metrics[k]['mae_eV'] for k in ['HOMO', 'LUMO', 'Gap']])), 'r2': float(np.mean([metrics[k]['r2'] for k in ['HOMO', 'LUMO', 'Gap']]))}
record = {'notebook': 'colab_schnet_3d_1m.ipynb', 'raw_csv': str(RAW_CSV), 'base_cache': str(BASE_CACHE), 'dataset_mode': DATASET_MODE, 'n_graphs': len(train_set)+len(val_set)+len(test_set), 'train_limit': TRAIN_LIMIT, 'split': {'seed': SEED, 'train': len(train_set), 'val': len(val_set), 'test': len(test_set)}, 'model_params': PARAMS, 'batch_size': BATCH_SIZE, 'epochs_completed': len(log_rows), 'best_epoch': best_epoch, 'best_val_mae_eV': best_val, 'test_metrics': metrics}
metrics_path.write_text(json.dumps(record, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))
print('best checkpoint:', best_path)
print('metrics        :', metrics_path)
print('train log      :', log_path)

In [ ]:
# 7. Export the best 3D embedding for late fusion. This follows every successful training run.
# The checkpoint SHA prevents a stale embedding from being reused after retraining.
EMBEDDINGS_PATH = RESULTS_ROOT / SCHNET_EMBEDDING_NAME

def checkpoint_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

checkpoint_hash = checkpoint_sha256(best_path)
current_source = torch.cat([graph.source_idx.view(-1).cpu() for graph in graphs])
reuse_embedding = False
if EMBEDDINGS_PATH.exists():
    try:
        existing = torch.load(EMBEDDINGS_PATH, map_location='cpu', weights_only=False)
        reuse_embedding = (
            existing.get('checkpoint_sha256') == checkpoint_hash
            and existing['embeddings'].shape == (len(graphs), 192)
            and torch.equal(existing['source_idx'].cpu(), current_source)
        )
    except Exception:
        reuse_embedding = False

if reuse_embedding:
    print('3D embedding already matches best checkpoint:', EMBEDDINGS_PATH)
else:
    embedding_loader = DataLoader(
        graphs, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=NUM_WORKERS > 0,
    )
    embeddings, embedding_source = [], []
    model.eval()
    with torch.no_grad():
        for index, batch in enumerate(embedding_loader, start=1):
            batch = batch.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                encoded = model.encode(batch.z, batch.pos, batch.batch, charges=getattr(batch, 'charges', None))
            embeddings.append(encoded.float().cpu())
            embedding_source.append(batch.source_idx.view(-1).cpu())
            if index % 250 == 0:
                print(f'3D embedding batch {index}/{len(embedding_loader)}', flush=True)
    payload = {
        'embeddings': torch.cat(embeddings),
        'source_idx': torch.cat(embedding_source),
        'checkpoint': str(best_path),
        'checkpoint_sha256': checkpoint_hash,
        'dataset_mode': DATASET_MODE,
    }
    assert payload['embeddings'].shape == (len(graphs), 192)
    assert torch.equal(payload['source_idx'], current_source)
    temporary = EMBEDDINGS_PATH.with_suffix('.pt.tmp')
    torch.save(payload, temporary)
    temporary.replace(EMBEDDINGS_PATH)
    print('3D embedding ->', EMBEDDINGS_PATH, tuple(payload['embeddings'].shape))
